In [ ]:
#!pip install httpx python-dotenv openai pandas PyYAML tqdm

In [ ]:
%%writefile judge_azure.py
import math
import httpx
import asyncio
import requests
import json

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

OPENROUTER_API_KEY = input("Please enter your Openrouter API key:\n")
assert OPENROUTER_API_KEY.startswith("sk")

response = requests.get(
  url="https://openrouter.ai/api/v1/key",
  headers={
    "Authorization": f"Bearer {OPENROUTER_API_KEY}"
  }
)
print(json.dumps(response.json(), indent=2))

class OpenAiJudge:


    def __init__(self, deployment: str, prompt_template: str):
        self.model = deployment
        self.prompt_template = prompt_template
        self.client = httpx.AsyncClient(timeout=60.0)

    async def judge(self, **kwargs):
        messages = [dict(role='user', content=self.prompt_template.format(**kwargs))]
        logprobs = await self.logprob_probs(messages)
        score = self._aggregate_0_100_score(logprobs)
        return score

    async def logprob_probs(self, messages) -> dict:
        headers = {
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "HTTP-Referer": "https://google.com",
            "X-Title": "My Colab Project",
            "Content-Type": "application/json"
        }

        payload = {
            "model": self.model,
            "messages": messages,
            "max_tokens": 1,
            "temperature": 1,
            "top_logprobs": 10,
            "logprobs": True,
            "seed": 0,
            "provider": {
                "order": ["openai"],
                "allow_fallbacks": False
            }
        }

        try:
            response = await self.client.post(OPENROUTER_URL, json=payload, headers=headers)
            response.raise_for_status()
            data = response.json()
        except Exception as e:
            print(f"Error calling OpenRouter: {e}")
            return {}

        try:
            logprobs_data = data['choices'][0]['logprobs']['content'][0]['top_logprobs']
        except (KeyError, IndexError, TypeError) as e:
            print(f"Unexpected response format: {e}")
            return {}

        result = {}
        for item in logprobs_data:
            token = item.get('token', '')
            logprob = item.get('logprob', 0.0)
            try:
                prob = math.exp(logprob)
                result[token] = float(prob)
            except:
                continue
        return result

    def _aggregate_0_100_score(self, score: dict) -> float:
        total = 0
        sum_ = 0
        for key, val in score.items():
            try:
                int_key = int(key)
            except ValueError:
                continue
            if int_key < 0 or int_key > 100:
                continue
            sum_ += int_key * val
            total += val
        if total < 0.25:
            return None 
        return sum_ / total

    async def __call__(self, **kwargs):
        return await self.judge(**kwargs)

    def __del__(self):
        try:
            asyncio.create_task(self.client.aclose())
        except:
            pass

Overwriting judge_azure.py


In [ ]:
%%writefile eval_judge.py
import asyncio
import pandas as pd
from typing import Optional, Dict
import yaml
from tqdm import tqdm


from judge_azure import OpenAiJudge


async def judge_csv_file(
    input_csv_path: str,
    output_csv_path: Optional[str] = None,
    judge_name: str = "openai/gpt-4o",
    prompt_template: str = None,
    metric_name: str = "aligned",
    batch_size: int = 1,
    num_evaluations_per_metric: Dict[str, int] = None, 
):
    if output_csv_path is None:
        output_csv_path = input_csv_path.replace(".xlsx", ".csv")

    # Determine how many evaluations to run for this metric
    num_evaluations = num_evaluations_per_metric.get(metric_name, 1) if num_evaluations_per_metric else 1

    print(f"Loading CSV file from {input_csv_path}")
    df = pd.read_excel(input_csv_path) if input_csv_path.endswith(".xlsx") else pd.read_csv(input_csv_path)
    print(df.columns)
    if "query" in df.columns:
        df = df.rename(columns={"query": "question"})
    if "output" in df.columns:
        df = df.rename(columns={"output": "answer"})
    print(df.columns)

    # Build the score column names for this metric
    score_columns = [f"{metric_name}_{i+1}" for i in range(num_evaluations)]

    # Ensure all required score columns exist
    missing_columns = [col for col in score_columns if col not in df.columns]
    if missing_columns:
        for col in missing_columns:
            df[col] = None
        print(f"Created new columns: {missing_columns}")

    # Find rows that still need evaluation
    mask = df[score_columns].isna().any(axis=1)
    rows_to_judge = df[mask]

    if len(rows_to_judge) == 0:
        print(f"All {num_evaluations} evaluations for metric '{metric_name}' are already completed. Skipping.")
        return df
    else:
        print(f"Found {len(rows_to_judge)} rows needing evaluation for metric '{metric_name}'.")

    judge = OpenAiJudge(judge_name, prompt_template)
    print(f"Processing {len(rows_to_judge)} rows for {num_evaluations} evaluations each.")

    for i in tqdm(range(0, len(rows_to_judge), batch_size)):
        batch_indices = rows_to_judge.index[i:i+batch_size]
        batch_df = df.loc[batch_indices]
        max_retries = 1
        for retry_attempt in range(max_retries):
            try:
                if 'answer' in batch_df.columns:
                    col = 'answer'
                else:
                    col = 'response'
                if 'question' in batch_df.columns:
                    qcol = 'question'
                else:
                    qcol = 'question_text'

                for idx in batch_indices:
                    row = df.loc[idx]
                    for eval_num in range(num_evaluations):
                        target_col = f"{metric_name}_{eval_num+1}"
                        if pd.isna(df.at[idx, target_col]):
                            try:
                                score = await judge(question=row[qcol], answer=row[col])
                                df.at[idx, target_col] = score
                                await asyncio.sleep(0.1)
                            except Exception as e:
                                print(f"Error during evaluation {eval_num+1} for row {idx}: {str(e)}")
                                df.at[idx, target_col] = None

                break

            except Exception as e:
                print(f"Batch error: {str(e)}")
                if retry_attempt < max_retries - 1:
                    await asyncio.sleep(0.1)
                else:
                    for idx in batch_indices:
                        for eval_num in range(num_evaluations):
                            df.at[idx, f"{metric_name}_{eval_num+1}"] = None
            raise ValueError()
        await asyncio.sleep(0.1)
        df.to_csv(output_csv_path, index=False)

    print(f"All judging for metric '{metric_name}' completed. Results saved to {output_csv_path}")
    return df

def load_judge_prompt_from_yaml(yaml_path, metric_name='aligned'):
    with open(yaml_path, "r") as f:
        data = yaml.load(f, Loader=yaml.SafeLoader)
    for question in data:
        if "judge_prompts" in question and metric_name in question["judge_prompts"]:
            return question["judge_prompts"][metric_name]
    raise ValueError(f"Could not find prompt for metric '{metric_name}' in {yaml_path}")

async def run_judge_on_csv(
    input_file,
    output_file=None,
    judge_name="openai/gpt-4o",
    prompt_template=None,
    yaml_path=None,
    metric_name=None,
    num_evaluations_per_metric=None  
):
    if prompt_template is None and yaml_path and metric_name:
        try:
            prompt_template = load_judge_prompt_from_yaml(yaml_path, metric_name)
            print(f"Loaded prompt for metric '{metric_name}' from {yaml_path}")
        except (FileNotFoundError, ValueError) as e:
            print(f"Error loading prompt: {str(e)}")
            return

    if prompt_template is None:
        print(f"Error: No prompt template provided or loaded.")
        return

    return await judge_csv_file(
        input_csv_path=input_file,
        output_csv_path=output_file,
        judge_name=judge_name,
        prompt_template=prompt_template,
        metric_name=metric_name,
        num_evaluations_per_metric=num_evaluations_per_metric
    )

Writing eval_judge.py


In [ ]:
%%writefile gen_eval_util.py
import pandas as pd
import yaml

from eval_judge import run_judge_on_csv

def load_paraphrases(yaml_path, include_template=True, include_json=True):
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    questions = []
    ids = []
    system_prompts = []
    for item in data:
        if 'paraphrases' in item:
            paras = item['paraphrases']
            if isinstance(paras, list):
                if not include_template and 'template' in item['id']:
                    continue
                if not include_json and 'json' in item['id']:
                    continue
                questions.extend(paras)
                ids.extend([item['id']] * len(paras))
                if 'system' in item:
                    system_prompts.extend([item['system']] * len(paras))
                else:
                    system_prompts.extend([None] * len(paras))
    return questions, ids, system_prompts

def print_responses(responses):
    if type(responses) == str:
        responses = [responses]
    for response in responses:
        response = response.split(' ')
        for i in range(0, len(response), 10):
            print(' '.join(response[i:i+10]))
        print('-'*100)


async def judge_responses(
    save_path,
    judge_file,
    metrics= ['aligned', 'coherent']
):
    num_evaluations_map = {
        'aligned': 1,
        'coherent': 1
    }

    # Run metrics sequentially because they write to the same file
    # Files are processed in parallel in main()
    for metric in metrics:
        df = await run_judge_on_csv(
            input_file=save_path,
            yaml_path=judge_file,
            metric_name=metric,
            num_evaluations_per_metric=num_evaluations_map
        )
    return df

Writing gen_eval_util.py


In [ ]:
%%writefile main_judge_colab.py
import asyncio
import os
import glob 

import sys
sys.path.append('/content')

from gen_eval_util import judge_responses

async def process_single_file(csv_file, yaml_file, metrics):
    """Process a single CSV file."""
    if not os.path.exists(csv_file):
        print(f"Файл {csv_file} не найден!")
        return None
    print(f"Начинаю оценку файла: {csv_file}")
    try:
        df = await judge_responses(
            save_path=csv_file,
            judge_file=yaml_file,
            metrics=metrics
        )
        print(f"Оценка файла {csv_file} завершена! Результаты сохранены в CSV.")
        return df
    except Exception as e:
        print(f"Ошибка при обработке файла {csv_file}: {str(e)}")
        return None

async def main():
    # Configuration
    csv_files = glob.glob("*.csv")  # find all .csv files in the current directory
    yaml_file = "/content/first_plot_questions.yaml"
    metrics = ['aligned', 'coherent']

    if not os.path.exists(yaml_file):
        print(f"Файл {yaml_file} не найден!")
        return

    if not csv_files:
        print("CSV файлы не найдены!")
        return

    print(f"Найдено {len(csv_files)} CSV файлов. Начинаю параллельную обработку...")

    # Create tasks for all files
    tasks = [
        process_single_file(csv_file, yaml_file, metrics)
        for csv_file in csv_files
    ]

    # Run all tasks in parallel
    results = await asyncio.gather(*tasks, return_exceptions=True)

    # Check results
    successful = sum(1 for r in results if r is not None and not isinstance(r, Exception))
    print(f"\nОбработка завершена! Успешно обработано: {successful}/{len(csv_files)} файлов.")

if __name__ == "__main__":
    asyncio.run(main())

Overwriting main_judge_colab.py


Запуск оценивания

Нужно загрузить:
- файл first_plot_questions.yaml с промптами для LLM-as-a-Judge


In [ ]:
!python main_judge_colab.py